In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Load dataset
data = pd.read_csv("train.csv")

# Display basic information
print(data.head())
print(data.info())

   Id  Elevation  Aspect  Slope  Horizontal_Distance_To_Hydrology  \
0   1       2596      51      3                               258   
1   2       2590      56      2                               212   
2   3       2804     139      9                               268   
3   4       2785     155     18                               242   
4   5       2595      45      2                               153   

   Vertical_Distance_To_Hydrology  Horizontal_Distance_To_Roadways  \
0                               0                              510   
1                              -6                              390   
2                              65                             3180   
3                             118                             3090   
4                              -1                              391   

   Hillshade_9am  Hillshade_Noon  Hillshade_3pm  ...  Soil_Type32  \
0            221             232            148  ...            0   
1            220          

In [3]:
# Separate features and target
X = data.drop("Cover_Type", axis=1)
y = data["Cover_Type"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=100,stratify=y)

In [5]:
rf_base = RandomForestClassifier(n_estimators=100,random_state=100,n_jobs=-1)

In [6]:
rf_base.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=100)

In [7]:
y_pred_base = rf_base.predict(X_test)

scr = accuracy_score(y_test,y_pred_base)
print(f'Accuracy: {scr * 100:.2f}%')

Accuracy: 86.90%


USING BEST PARAMETERS

In [8]:
# Define Random Forest model
rf = RandomForestClassifier(random_state=100,n_jobs=-1)

In [9]:
# Parameter grid for tuning
param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [None, 30, 50],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

In [10]:
# GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

In [11]:
# Train GridSearchCV
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


GridSearchCV(cv=3,
             estimator=RandomForestClassifier(n_jobs=-1, random_state=100),
             n_jobs=-1,
             param_grid={'max_depth': [None, 30, 50],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [200, 300]},
             scoring='accuracy', verbose=2)

In [13]:
# Best model
best_rf = grid_search.best_estimator_

In [14]:
print("Best Parameters Found:")
print(grid_search.best_params_)

Best Parameters Found:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


In [15]:
# Predictions using tuned model
y_pred = best_rf.predict(X_test)

In [16]:
# Accuracy
scr = accuracy_score(y_test,y_pred)
print(f'Accuracy: {scr * 100:.2f}%')

Accuracy: 86.94%


In [17]:
# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           1       0.81      0.74      0.78       432
           2       0.80      0.73      0.76       432
           3       0.86      0.82      0.84       432
           4       0.93      0.98      0.95       432
           5       0.90      0.95      0.92       432
           6       0.84      0.90      0.87       432
           7       0.93      0.96      0.95       432

    accuracy                           0.87      3024
   macro avg       0.87      0.87      0.87      3024
weighted avg       0.87      0.87      0.87      3024



In [18]:
# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:

[[320  68   1   0  12   1  30]
 [ 56 314  16   0  28  17   1]
 [  0   1 356  22   4  49   0]
 [  0   0   5 422   0   5   0]
 [  0  10   7   0 412   3   0]
 [  0   1  28   9   4 390   0]
 [ 17   0   0   0   0   0 415]]


In [19]:
feature_importance = pd.Series(
    best_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance.head(10))


Elevation                             0.207441
Horizontal_Distance_To_Roadways       0.084400
Id                                    0.080725
Horizontal_Distance_To_Fire_Points    0.065111
Horizontal_Distance_To_Hydrology      0.055504
Vertical_Distance_To_Hydrology        0.050777
Hillshade_9am                         0.046442
Aspect                                0.045053
Wilderness_Area4                      0.042816
Hillshade_3pm                         0.041146
dtype: float64
